# Chapter 27 — Debugging AI-Generated Designs

**Book alignment:** Debugging AI From First Principles, Chapter 27

**Question this notebook isolates:** The sequence diagram is "clean"; the p99 budget is
200 ms and the hops sum to 450. Does a **constraint table** (bound + line-item arithmetic +
verdict per row) plus a **tradeoff matrix** (≥2 priced candidates) separate **H1**
(constraint violation under the design's own numbers), **H2** (constraint unverifiable —
bound unstated), and **H3** (tradeoff blindness — one shape, no priced alternative)?

In [ ]:
DESIGN = {
    "hops_ms": {"auth": 80, "profile": 190, "reco": 180},   # the design's OWN stated per-hop p99
    "fan_out_reads": 3,
    "consistency": "async invalidation",                    # no staleness bound stated
    "reads_pinned_single_az": True,
}
BOUNDS = {"p99_ms": 200, "peak_rps": 100, "downstream_quota_rps": 250, "az_loss_tolerable": True}

## 1. The constraint table — arithmetic, executed, not asserted

In [ ]:
rows = []
seq_latency = sum(DESIGN["hops_ms"].values())
rows.append(("p99 latency", f"< {BOUNDS['p99_ms']}ms",
             f"{' + '.join(map(str, DESIGN['hops_ms'].values()))} = {seq_latency}ms",
             "FAIL (H1)" if seq_latency > BOUNDS["p99_ms"] else "PASS"))

downstream = DESIGN["fan_out_reads"] * BOUNDS["peak_rps"]
rows.append(("peak load", f"<= {BOUNDS['downstream_quota_rps']} rps",
             f"{DESIGN['fan_out_reads']} x {BOUNDS['peak_rps']} = {downstream} rps",
             "FAIL (H1)" if downstream > BOUNDS["downstream_quota_rps"] else "PASS"))

rows.append(("consistency", "read-your-write, bound stated", DESIGN["consistency"] + " (no bound)",
             "UNVERIFIABLE (H2)"))
rows.append(("blast radius", "1 AZ loss tolerable",
             "all reads single-AZ pinned" if DESIGN["reads_pinned_single_az"] else "multi-AZ",
             "FAIL (H1)" if DESIGN["reads_pinned_single_az"] else "PASS"))

for name, bound, items, verdict in rows:
    print(f"{name:14} | {bound:28} | {items:24} | {verdict}")

fails = [r for r in rows if "FAIL" in r[3]]
unver = [r for r in rows if "UNVERIFIABLE" in r[3]]
assert len(fails) >= 2 and len(unver) >= 1
print("\none FAIL row rejects the design SHAPE; one UNVERIFIABLE row rejects the design DOCUMENT")

## 2. The tradeoff matrix — >=2 candidates priced on identical rows

In [ ]:
CANDIDATES = {
    "A 3-way fan-out (proposed)": dict(p99=450, load=300, consistency="unbounded"),
    "B cached + collapsed hops":  dict(p99=120, load=110, consistency="30s staleness, versioned"),
    "C sync write-through":       dict(p99=260, load=200, consistency="read-your-write"),
}
for name, c in CANDIDATES.items():
    ok = c["p99"] <= BOUNDS["p99_ms"] and c["load"] <= BOUNDS["downstream_quota_rps"]
    print(f"{name:28} p99 {c['p99']:>4}ms  load {c['load']:>4}rps  -> {'VIABLE' if ok else 'REJECT'}")
assert CANDIDATES["A 3-way fan-out (proposed)"]["p99"] > BOUNDS["p99_ms"]
assert CANDIDATES["B cached + collapsed hops"]["p99"] <= BOUNDS["p99_ms"]
print("\nH3 exonerated once B is priced - the alternative existed; the COMPARISON was missing")

## 3. Why execute the arithmetic instead of trusting the prose

In [ ]:
design_prose = "Parallel fan-out keeps latency flat and well within budget."
# the prose is a compositional claim (add hops, account for overlap/queueing) reasoned in prose.
# executed: even fully parallel, the floor is max(hops) + overhead, and no overlap was stated.
parallel_floor = max(DESIGN["hops_ms"].values())
print(f"prose: {design_prose!r}")
print(f"executed: sequential {seq_latency}ms ; parallel floor {parallel_floor}ms ; budget {BOUNDS['p99_ms']}ms")
assert parallel_floor <= BOUNDS["p99_ms"] < seq_latency   # even the best case needs stated overlap to pass
print("demand the computation shown, not asserted - transformers are unreliable at exactly this composition")

## What we earned

A design is a constraint-satisfaction claim, debugged like a proof, line by line. The
constraint table priced the design's *own* numbers: latency 450 > 200 and load 300 > 250
both **FAIL (H1)**, consistency unbounded is **UNVERIFIABLE (H2)**. One FAIL row rejects the
shape; one UNVERIFIABLE row rejects the document — for the cost of an envelope, before any
staging environment. The tradeoff matrix (candidate B priced and viable) retired H3. The
design's "within budget" prose was a compositional claim reasoned in prose; demand it
executed.

**Notebook 28 / Chapter 28** takes the priced design's "related work" section, where the
prose is about sources rather than systems.